# ChuckleNet: Full Pipeline (FIXED)
## Prosody extraction + Training for 555 videos

**Pipeline:**
1. Load WavLM embeddings (video_id + start/end + embedding)
2. Parse VTT files for utterance boundaries + [laughter] labels
3. Match WavLM to utterances by (video_id, start, end)
4. Extract 21-dim prosody from audio
5. Train fusion model with ALL fixes

**Runtime:** ~60-90 min on T4

In [ ]:
# @title Step 1: Install deps
!pip install -q kaggle librosa scikit-learn

In [ ]:
# @title Step 2: Download data from Kaggle
import os
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

# WavLM embeddings
!kaggle datasets download -d subhajitdas/chuckle-wavlm-555-videos -p {DATA_DIR} --unzip -q

# VTT files with labels
!kaggle datasets download -d subhajitdas/chuckle-vtt-labels -p {DATA_DIR} --unzip -q

# Audio (only what's needed)
# !kaggle datasets download -d subhajitdas/chuckle-vtt-audio-tar -p {DATA_DIR} --unzip -q

print('✅ Data downloaded')
!ls -la

In [ ]:
# @title Step 3: Parse VTT files for labels
import re
from pathlib import Path

def parse_vtt_label(vtt_path):
    """Parse VTT file to extract utterance labels."""
    with open(vtt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract video_id from filename (remove .vtt)
    video_id = Path(vtt_path).stem
    
    labels = {}
    
    # VTT format: timestamp lines followed by text
    # Example:
    # 00:00:01.420 --> 00:00:03.590
    # Some text [laughter] here
    
    # Pattern to match timestamp lines
    timestamp_pattern = r'(\d{2}:\d{2}:\d{2}\.\d{3})\s-->\s(\d{2}:\d{2}:\d{2}\.\d{3})'
    
    lines = content.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # Check if this line is a timestamp
        match = re.match(timestamp_pattern, line)
        if match:
            start_ts = match.group(1)
            end_ts = match.group(2)
            
            # Convert to seconds
            def ts_to_sec(ts):
                parts = ts.split(':')
                return float(parts[0])*3600 + float(parts[1])*60 + float(parts[2])
            
            start = ts_to_sec(start_ts)
            end = ts_to_sec(end_ts)
            
            # Get text (may be on next lines)
            i += 1
            text_parts = []
            while i < len(lines) and lines[i].strip() and not re.match(timestamp_pattern, lines[i].strip()):
                text_parts.append(lines[i].strip())
                i += 1
            
            text = ' '.join(text_parts)
            
            # Check for [laughter] marker
            has_laughter = '[laughter]' in text.lower()
            
            # Store with (start, end) as key
            key = (round(start, 2), round(end, 2))
            labels[key] = 1 if has_laughter else 0
            continue
        i += 1
    
    return video_id, labels

# Parse all VTT files
print('Parsing VTT files...')
vtt_dir = Path(f'{DATA_DIR}/chuckle-vtt-labels')
vtt_files = list(vtt_dir.glob('*.vtt'))
print(f'Found {len(vtt_files)} VTT files')

all_labels = {}  # video_id -> { (start,end): label }
for vtt_file in vtt_files:
    vid, labels = parse_vtt_label(vtt_file)
    if labels:
        all_labels[vid] = labels

print(f'Parsed labels for {len(all_labels)} videos')
print(f'Sample: {list(all_labels.items())[0] if all_labels else "None"}')

In [ ]:
# @title Step 4: Load WavLM embeddings
import json
from pathlib import Path

WAVLM_DIR = Path(f'{DATA_DIR}/chuckle-wavlm-555-videos')

wavlm_data = {}  # video_id -> [ {start, end, embedding}, ... ]
for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    
    if 'embeddings' in data:
        wavlm_data[vid] = data['embeddings']

print(f'Loaded WavLM for {len(wavlm_data)} videos')
total = sum(len(v) for v in wavlm_data.values())
print(f'Total utterances: {total}')

In [ ]:
# @title Step 5: Match WavLM with VTT labels
matched = 0
unmatched = 0
pos_count = 0

for vid, embeddings in wavlm_data.items():
    if vid not in all_labels:
        # No VTT file for this video - skip
        for emb in embeddings:
            emb['label'] = 0
            unmatched += 1
        continue
    
    labels_dict = all_labels[vid]
    
    for emb in embeddings:
        # Match by rounded start/end
        key = (round(emb['start'], 2), round(emb['end'], 2))
        
        if key in labels_dict:
            emb['label'] = labels_dict[key]
            matched += 1
            if emb['label'] == 1:
                pos_count += 1
        else:
            # Try fuzzy match with small tolerance
            found = False
            for (s, e), l in labels_dict.items():
                if abs(s - emb['start']) < 0.5 and abs(e - emb['end']) < 0.5:
                    emb['label'] = l
                    matched += 1
                    if l == 1:
                        pos_count += 1
                    found = True
                    break
            if not found:
                emb['label'] = 0
                unmatched += 1

total_utts = sum(len(v) for v in wavlm_data.values())
neg_count = total_utts - matched - unmatched + pos_count

print(f'Matched: {matched}')
print(f'Unmatched (no VTT): {unmatched}')
print(f'Positive: {pos_count} ({pos_count/total_utts*100:.1f}%)')
print(f'Negative: {neg_count} ({neg_count/total_utts*100:.1f}%)')

In [ ]:
# @title Step 6: Extract prosody (21-dim)
import numpy as np
import librosa
import time
from pathlib import Path

SR = 16000

# Mount audio from Google Drive (needed for prosody)
from google.colab import drive
drive.mount('/content/gdrive')

AUDIO_BASE = Path('/content/gdrive/MyDrive')

def extract_prosody_21dim(y, sr):
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms) - np.min(rms)])
    
    # Duration - 2 dims
    features.extend([len(y) / sr, len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat), np.mean(zcr), np.std(zcr)])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

def get_audio_path(vid):
    """Find audio for video in Google Drive."""
    # Search all audio folders
    for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 
                   'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new',
                   'chuckle_audio_all/audio_all']:
        audio_dir = AUDIO_BASE / folder
        if audio_dir.exists():
            for ext in ['.wav', '.mp3', '.m4a']:
                p = audio_dir / f'{vid}{ext}'
                if p.exists():
                    return str(p)
    return None

print('Extracting prosody...')
t0 = time.time()

prosody_data = {}  # video_id -> [np.array(21), ...]
failed = []

for i, (vid, embeddings) in enumerate(wavlm_data.items()):
    audio_path = get_audio_path(vid)
    
    if not audio_path:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32)] * len(embeddings)
        continue
    
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True)
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        video_prosody = []
        for emb in embeddings:
            start_sample = int(emb['start'] * SR)
            end_sample = int(emb['end'] * SR)
            if end_sample > len(y):
                end_sample = len(y)
            y_slice = y[start_sample:end_sample]
            
            if len(y_slice) < SR * 0.1:
                video_prosody.append(np.zeros(21, dtype=np.float32))
            else:
                prosody = extract_prosody_21dim(y_slice, SR)
                video_prosody.append(prosody)
        
        prosody_data[vid] = video_prosody
        
    except Exception as e:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32)] * len(embeddings)
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'{i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f}min | Failed: {len(failed)}')

print(f'\n✅ Done! Prosody: {len(prosody_data)} | Failed: {len(failed)}')
print(f'Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# @title Step 7: Prepare combined data
all_emb = []
all_pros = []
all_labels_list = []

for vid, embeddings in wavlm_data.items():
    if vid not in prosody_data:
        continue
    for i, emb in enumerate(embeddings):
        all_emb.append(emb['embedding'])
        all_pros.append(prosody_data[vid][i])
        all_labels_list.append(emb.get('label', 0))

all_emb = np.array(all_emb, dtype=np.float32)
all_pros = np.array(all_pros, dtype=np.float32)
all_labels_arr = np.array(all_labels_list, dtype=np.int64)

print(f'Total: {len(all_emb)} samples')
pos = sum(all_labels_list)
print(f'Positive: {pos} ({pos/len(all_labels_list)*100:.1f}%)')
print(f'WavLM: {all_emb.shape}, Prosody: {all_pros.shape}')

In [ ]:
# @title Step 8: Train/Val/Test split + Training (ALL FIXES)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Split
X_train, X_test, pros_train, pros_test, y_train, y_test = train_test_split(
    all_emb, all_pros, all_labels_arr, test_size=0.2, random_state=42, stratify=all_labels_arr
)
X_train, X_val, pros_train, pros_val, y_train, y_val = train_test_split(
    X_train, pros_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

# DataLoaders
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(pros_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(pros_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(pros_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# @title Step 9: Model + Training (ALL FIXES)
class FusionModel(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_feat = self.prosody_proj(prosody)
        x = torch.cat([wavlm_emb, prosody_feat], dim=-1)
        return self.classifier(x)

model = FusionModel().to(device)

# FIXED: Class weights [1.0, 2.5] + CrossEntropyLoss
class_weights = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for emb_b, pros_b, labels_b in train_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        labels_b = labels_b.to(device)
        
        optimizer.zero_grad()
        logits = model(emb_b, pros_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # FIXED
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for emb_b, pros_b, labels_b in val_loader:
            emb_b = emb_b.to(device)
            pros_b = pros_b.to(device)
            logits = model(emb_b, pros_b)
            preds = torch.argmax(logits, dim=-1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels_b.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, '/content/best_model.pt')
        print(f'  ✅ New best!')

In [ ]:
# @title Step 10: Final Evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for emb_b, pros_b, labels_b in test_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        logits = model(emb_b, pros_b)
        preds = torch.argmax(logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_b.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'\n🏆 Test F1: {test_f1:.4f}')
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))

print('\n✅ Done!')